In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_fi = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/fi_2016_2022_bn.csv")

In [3]:
df_fi = df_fi.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_fi

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,DI,COC,CPI,ROL,VAA,EUPC
0,2.50,5.7,5.8,2.33,143.000000,0.78,20.23,21.680712,24.942915,2.271802,...,72.5,0.04,-548.050,1635.000000,9.20,2.20,90.0,1.88,1.54,4.014785e+05
1,1.09,5.5,6.1,2.11,144.000000,0.64,20.42,21.956824,24.449662,2.395591,...,73.2,0.03,3409.890,-2276.000000,9.22,2.19,88.0,1.80,1.52,5.817749e+06
2,2.47,5.5,5.8,2.19,145.000000,0.50,20.58,22.532315,24.381107,1.356196,...,73.9,0.02,-1035.440,-555.000000,9.22,2.16,88.0,1.77,1.57,6.781235e+06
3,-1.10,5.6,5.8,2.22,145.000000,0.36,20.64,21.886000,24.204777,1.347792,...,74.7,0.04,2311.730,-3538.000000,9.22,2.12,87.0,1.83,1.54,8.928446e+06
4,0.36,5.5,5.8,2.11,146.000000,0.29,19.56,22.762653,24.679988,-2.066138,...,74.3,0.00,372.670,-2434.000000,9.15,2.24,88.0,1.81,1.51,8.953195e+06
5,4.16,5.5,5.8,2.04,146.000000,0.43,19.80,23.602002,23.845550,6.917664,...,75.5,0.05,2650.040,-2409.928571,9.09,2.33,88.0,1.90,1.54,1.380688e+07
6,7.67,5.5,5.8,2.12,141.466667,0.79,20.84,24.693918,21.681084,0.743682,...,77.4,0.07,1126.794,-2409.928571,9.28,2.40,90.0,1.90,1.59,1.271747e+07


In [5]:
discretised_fi = pd.DataFrame(index=df_fi.index)

for col in df_fi.columns:
    no_unique = df_fi[col].nunique()
    
    if no_unique <= 1:
        discretised_fi[col] = 0
    else:
        try:
            discretised_fi[col] = pd.qcut(
                df_fi[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_fi[col] = df_fi[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_fi)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    1    0    0     2     0    2     0     0     2      1  ...    0    1   
1    0    0    0     0     0    1     1     0     1      2  ...    0    0   
2    1    0    0     1     1    1     1     1     1      1  ...    0    0   
3    0    0    0     2     1    0     2     0     0      0  ...    1    1   
4    0    0    0     0     2    0     0     1     2      0  ...    1    0   
5    2    0    0     0     2    0     0     2     0      2  ...    2    2   
6    2    0    0     1     0    2     2     2     0      0  ...    2    2   

   EPFRG  GGAG  DI  COC  CPI  ROL  VAA  EUPC  
0      0     2   0    1    1    1    0     0  
1      2     1   1    0    0    0    0     0  
2      0     2   1    0    0    0    1     0  
3      1     0   1    0    0    1    0     1  
4      0     0   0    1    0    0    0     1  
5      2     0   0    2    0    2    0     2  
6      1     0   2    2    1    2   

In [6]:
sm = from_pandas(discretised_fi)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_fi.html")

Graphs/01_fully_connected_dk.html


In [11]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_fi.html")

Graphs/01_thresholded_dk.html


In [12]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_fi_1.html")

In [13]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

     From     To    Weight
27    COC   PDPS  3.197870
4     PGP   GGAG  2.528513
29    COC   GGFC  2.303506
39    CPI     DI  1.753891
22    LFG   EIAP  1.691235
34    CPI    PGP  1.561108
20   GDPC     DI  1.503061
11    EII   EBSG  1.347220
26  EPFRG   LBGD  1.299657
30    COC    EIS  1.224153
7     EII    FDI  1.107633
0     FDI   GGAG  1.093587
10    EII  GDPCG  1.093093
13    EII  EPFRG  1.068320
1    EIAP    EIS  1.041282
23    LFG   PDPS  1.014170
2    PDPS    LTU  1.013076
5     PGP    VAA  1.009286
41    VAA   GCFP  1.004833
31    COC   GGAG  0.967836
15    EII     DI  0.955993
3     PGP  EPFRG  0.948363
17   GDPC   VABI  0.936417
21    EIS   GGFC  0.835451
35    CPI  GDPCG -0.809832
14    EII   GGAG -0.854522
6    GCFP   GGAG -1.024752
24    LFG    EIS -1.078451
12    EII    EIS -1.087624
19   GDPC    LTU -1.308704
25    LFG     DI -1.337790
40    CPI    VAA -1.395671
28    COC   VABI -1.605910
36    CPI    EIS -1.609553
18   GDPC   GGFC -1.616685
16   GDPC   PDPS -1.661849
8